1. IMPORT LIBRARY

In [6]:
import pandas as pd
import numpy as np

2. LOAD DATA

In [7]:
df = pd.read_csv('modified_data.csv')
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft
0,2014-05-02,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,233.58
1,2014-05-02,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,653.15
2,2014-05-02,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,177.20
3,2014-05-02,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,210.00
4,2014-05-02,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,283.51


3. DROP UNUSED COLUMNS

In [8]:
df = df.drop(columns=['date', 'street', 'statezip', 'price_per_sqft'])
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,city
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,Shoreline
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,Seattle
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,Kent
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,Bellevue
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,Redmond


4. CREATE FEATURE HOUSE AGE

In [9]:
df['house_age'] = 2014 - df['yr_built']
df['was_renovated'] = (df['yr_renovated'] > 0).astype(int)
df['years_since_renovation'] = np.where(
    df['yr_renovated'] > 0,
    2014 - df['yr_renovated'],
    df['house_age']
)
df = df.drop(columns=['yr_built', 'yr_renovated'])
df.head()


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,city,house_age,was_renovated,years_since_renovation
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,Shoreline,59,1,9
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,Seattle,93,0,93
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,Kent,48,0,48
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,Bellevue,51,0,51
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,Redmond,38,1,22


5. ENCODING COLUMN CITY

In [10]:
city_freq = df['city'].value_counts(normalize=True)
df['city_encoded'] = df['city'].map(city_freq)
df = df.drop(columns=['city'])
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,house_age,was_renovated,years_since_renovation,city_encoded
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,59,1,9,0.026739
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,93,0,93,0.341957
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,48,0,48,0.040217
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,51,0,51,0.062174
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,38,1,22,0.051087


6. HANDLING OUTLIERS IN price and sqrt_living

In [11]:
def remove_outliers_iqr(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return data[(data[column] >= lower) & (data[column] <= upper)]

before = df.shape[0]
df = remove_outliers_iqr(df, 'price')
df = remove_outliers_iqr(df, 'sqft_living')
after = df.shape[0]
print(f'Baris sebelum: {before}, sesudah: {after}, dibuang: {before - after}')


Baris sebelum: 4600, sesudah: 4284, dibuang: 316


7. SAVE THE RESULT PROCESSING DATA

In [12]:
df.to_csv('processed_data.csv', index=False)
df.shape


(4284, 15)